# Day 047 — Exercise 4: compare_groups

**What you'll build:** `compare_groups(a, b, alpha=0.05) -> dict` — independent-samples t-test (`scipy.stats.ttest_ind`) plus Cohen's d effect size, returning whether the groups are statistically different and how large the difference is.

**Why it matters:** Knowing two group means differ is one thing; knowing *how much* they differ (effect size) is another. Cohen's d tells you if the difference is negligible (|d| < 0.2), small (0.2–0.5), medium (0.5–0.8), or large (> 0.8) — independent of sample size.

## Provided: Setup + all prior functions

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

def make_sample_data(n: int = 100, seed: int = 42) -> pd.DataFrame:
    """Return a reproducible multi-column dataset for statistics exercises."""
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        'normal_col': rng.standard_normal(n).round(3),
        'skewed_col': rng.exponential(2, n).round(3),
        'score_a':    (50 + rng.standard_normal(n) * 10).round(1),
        'score_b':    (70 + rng.standard_normal(n) * 10).round(1),
    })


def describe_distribution(series: pd.Series) -> dict:
    s   = series.dropna()
    q25 = float(s.quantile(0.25))
    q75 = float(s.quantile(0.75))
    return {
        'count':    int(len(s)),
        'mean':     round(float(s.mean()), 4),
        'median':   round(float(s.median()), 4),
        'std':      round(float(s.std(ddof=1)), 4),
        'sem':      round(float(s.sem()), 4),
        'min':      round(float(s.min()), 4),
        'max':      round(float(s.max()), 4),
        'q25':      round(q25, 4),
        'q75':      round(q75, 4),
        'iqr':      round(q75 - q25, 4),
        'skewness': round(float(s.skew()), 4),
        'kurtosis': round(float(s.kurt()), 4),
    }


def test_normality(series: pd.Series, alpha: float = 0.05) -> dict:
    s      = series.dropna()
    stat, p = stats.shapiro(s)
    return {
        'n':          len(s),
        'statistic':  round(float(stat), 4),
        'p_value':    round(float(p), 6),
        'is_normal':  bool(p > alpha),
        'alpha':      alpha,
    }


def correlation_with_pvalue(x: pd.Series, y: pd.Series,
                             method: str = 'pearson') -> dict:
    mask  = x.notna() & y.notna()
    x_c, y_c = x[mask], y[mask]
    if method == 'pearson':
        r, p = stats.pearsonr(x_c, y_c)
    elif method == 'spearman':
        r, p = stats.spearmanr(x_c, y_c)
    else:
        raise ValueError(f"method must be 'pearson' or 'spearman', got {method!r}")
    return {
        'method':         method,
        'n':              len(x_c),
        'r':              round(float(r), 4),
        'p_value':        round(float(p), 6),
        'is_significant': bool(p < 0.05),
    }

## Your Implementation

In [ ]:
def compare_groups(a: pd.Series, b: pd.Series,
                   alpha: float = 0.05) -> dict:
    """
    Independent-samples t-test + Cohen's d effect size.

    Args:
        a, b:  two numeric pd.Series to compare (NaN values are dropped)
        alpha: significance level (default 0.05)
    Returns:
        dict with keys: n_a, n_b, mean_a, mean_b, statistic, p_value,
                        is_significant, cohens_d, conclusion
        conclusion: 'different' if p_value < alpha, else 'not_different'
        cohens_d: (mean_a - mean_b) / pooled_std (signed)
    """
    a_c, b_c = a.dropna(), b.dropna()
    # TODO: t, p = stats.ttest_ind(a_c, b_c)
    # TODO: n_a, n_b = len(a_c), len(b_c)
    # TODO: std_a = float(a_c.std(ddof=1))
    # TODO: std_b = float(b_c.std(ddof=1))
    # TODO: pooled_var = ((n_a - 1) * std_a**2 + (n_b - 1) * std_b**2) / (n_a + n_b - 2)
    # TODO: pooled = np.sqrt(pooled_var) if pooled_var > 0 else 0.0
    # TODO: d = (float(a_c.mean()) - float(b_c.mean())) / pooled if pooled > 0 else 0.0
    # TODO: sig = bool(p < alpha)
    # TODO: return {
    #     'n_a': n_a, 'n_b': n_b,
    #     'mean_a': round(float(a_c.mean()), 4),
    #     'mean_b': round(float(b_c.mean()), 4),
    #     'statistic':      round(float(t), 4),
    #     'p_value':        round(float(p), 6),
    #     'is_significant': sig,
    #     'cohens_d':       round(d, 4),
    #     'conclusion':     'different' if sig else 'not_different',
    # }
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined, returns dict
    try:
        assert 'compare_groups' in globals()
        a = pd.Series([4.9, 5.1, 5.0, 5.05, 4.95])
        b = pd.Series([5.0, 4.95, 5.05, 5.0, 5.02])
        result = compare_groups(a, b)
        assert isinstance(result, dict), \
            f'expected dict, got {type(result).__name__}'
        passed += 1; print('\u2705 Check 1: compare_groups returns dict')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: all required keys
    try:
        for k in ('n_a', 'n_b', 'mean_a', 'mean_b', 'statistic',
                  'p_value', 'is_significant', 'cohens_d', 'conclusion'):
            assert k in result, f'missing key: {k!r}'
        passed += 1; print('\u2705 Check 2: all required keys present')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: conclusion is 'different' or 'not_different'
    try:
        assert result['conclusion'] in ('different', 'not_different'), \
            f'conclusion must be different/not_different, got {result["conclusion"]!r}'
        passed += 1; print(f'\u2705 Check 3: conclusion={result["conclusion"]!r}')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: similar groups → not_different
    try:
        assert result['conclusion'] == 'not_different', \
            f'groups with mean≈5.0 should be not_different (p={result["p_value"]}), got {result["conclusion"]!r}'
        passed += 1; print(f'\u2705 Check 4: similar groups → not_different (p={result["p_value"]:.4f})')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: very different groups → different
    try:
        df = make_sample_data(100)
        r5 = compare_groups(df['score_a'], df['score_b'])
        assert r5['conclusion'] == 'different', \
            f'score_a (mean≈50) vs score_b (mean≈70) should be different, p={r5["p_value"]}'
        assert abs(r5['cohens_d']) > 1.0, \
            f'large effect expected (|d|>1), got {r5["cohens_d"]}'
        passed += 1; print(f'\u2705 Check 5: different groups → different (d={r5["cohens_d"]:.2f})')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def compare_groups(a: pd.Series, b: pd.Series,
                   alpha: float = 0.05) -> dict:
    a_c, b_c  = a.dropna(), b.dropna()
    t, p       = stats.ttest_ind(a_c, b_c)
    n_a, n_b   = len(a_c), len(b_c)
    std_a      = float(a_c.std(ddof=1))
    std_b      = float(b_c.std(ddof=1))
    pooled_var = ((n_a - 1) * std_a**2 + (n_b - 1) * std_b**2) / (n_a + n_b - 2)
    pooled     = np.sqrt(pooled_var) if pooled_var > 0 else 0.0
    d          = (float(a_c.mean()) - float(b_c.mean())) / pooled if pooled > 0 else 0.0
    sig        = bool(p < alpha)
    return {
        'n_a':            n_a,
        'n_b':            n_b,
        'mean_a':         round(float(a_c.mean()), 4),
        'mean_b':         round(float(b_c.mean()), 4),
        'statistic':      round(float(t), 4),
        'p_value':        round(float(p), 6),
        'is_significant': sig,
        'cohens_d':       round(d, 4),
        'conclusion':     'different' if sig else 'not_different',
    }
```

</details>